# CellXGene V2 EDA

We have a S3 bucket with the CellXGene V2 AnnData files loaded.  In this notebook we take a look at the metadata and figure out how to select our training and test sets. 

In [1]:
%run notebook_setup.ipynb

2025-10-21 17:28:36 - autoreload enabled
2025-10-21 17:28:38 - repo_dir set to /Users/rj/personal/GenePT-tools
2025-10-21 17:28:42 - data_dir set to /Users/rj/personal/GenePT-tools/data


File already exists at /Users/rj/personal/GenePT-tools/data/GenePT_emebdding_v2.zip
Extracting files...
Extracting GenePT_emebdding_v2/
Skipping GenePT_emebdding_v2/NCBI_UniProt_summary_of_genes.json - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_embedding_ada_text.pickle - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_protein_embedding_model_3_text.pickle. - already exists with same size
Skipping GenePT_emebdding_v2/NCBI_summary_of_genes.json - already exists with same size
Extraction complete!
Skipping embedding_original_ada_text.parquet - already exists
Skipping embedding_original_large_3.parquet - already exists
Skipping embedding_associations_age_cell_type_drugs_pathways_openai_large.parquet - already exists
Skipping embedding_associations_age_drugs_pathways_openai_large.parquet - already exists
Skipping embedding_associations_cell_type_openai_large.parquet - already exists
Skipping embedding_associations_cell_type_tissue_drug_pat

# Load metadata about our dataset

I used https://github.com/honicky/anndata-metadata to extract metadata about our dataset (including cell-type statistics).  I saved it
to a Parquet file, so lets load it and see what we can learn about our dataset

In [23]:
import pandas as pd

metadata_pdf = pd.read_parquet(data_dir / "cellxgene_v2_metadata_v3.parquet")

In [24]:
from collections import Counter

# Flatten all lists in obs_contents into a single list
all_strings = [item for sublist in metadata_pdf.obs_contents for item in sublist]

# Count occurrences
string_counts = Counter(all_strings)

# To see the most common strings
print(string_counts.most_common())    

[('assay', 961), ('assay_ontology_term_id', 961), ('cell_type', 961), ('cell_type_ontology_term_id', 961), ('development_stage', 961), ('development_stage_ontology_term_id', 961), ('disease', 961), ('disease_ontology_term_id', 961), ('donor_id', 961), ('is_primary_data', 961), ('observation_joinid', 961), ('organism', 961), ('organism_ontology_term_id', 961), ('self_reported_ethnicity', 961), ('self_reported_ethnicity_ontology_term_id', 961), ('sex', 961), ('sex_ontology_term_id', 961), ('suspension_type', 961), ('tissue', 961), ('tissue_ontology_term_id', 961), ('tissue_type', 961), ('_index', 486), ('n_genes', 251), ('author_cell_type', 239), ('sample_id', 205), ('nCount_RNA', 202), ('nFeature_RNA', 202), ('n_counts', 199), ('dissection', 166), ('cluster_id', 161), ('CellID', 155), ('cell_cycle_score', 155), ('fraction_mitochondrial', 155), ('fraction_unspliced', 155), ('total_UMIs', 155), ('total_genes', 155), ('index', 155), ('roi', 139), ('subcluster_id', 137), ('supercluster_term

# What tissues are in our dataset

In [30]:
import json

def aggregate_obs_counts(pdf, obs_count_name):
  all_obs_counts = {}
  for json_obj in pdf.obs_counts:
    obs_counts = json.loads(json_obj[obs_count_name])
    for key, value in obs_counts.items():
      
      if key in all_obs_counts:
        all_obs_counts[key] += value
      else:
        all_obs_counts[key] = value

  obs_counts_pdf = pd.DataFrame(all_obs_counts.items(), columns=[obs_count_name, 'count']).set_index(obs_count_name).sort_values('count', ascending=False)
  return obs_counts_pdf

tissue_counts_pdf = aggregate_obs_counts(metadata_pdf, 'tissue')
tissue_counts_pdf.head(20)

,count
tissue,
blood,10841680
lung,6166950
breast,5633483
dorsolateral prefrontal cortex,3642741
middle temporal gyrus,3225610
cerebral cortex,3124314
liver,1894601
hippocampal formation,1794600
heart left ventricle,1764391


In [36]:
ontology_counts_pdf = aggregate_obs_counts(metadata_pdf, 'tissue')
ontology_counts_pdf['ontology_label'] = aggregate_obs_counts(metadata_pdf, 'tissue_ontology_term_id').index.str.split(':').str[0]
ontology_counts_pdf[ontology_counts_pdf.ontology_label == 'CL']

ontology_counts_pdf.head(20)

,count,ontology_label
tissue,,
blood,10841680,UBERON
lung,6166950,UBERON
breast,5633483,UBERON
dorsolateral prefrontal cortex,3642741,UBERON
middle temporal gyrus,3225610,UBERON
cerebral cortex,3124314,UBERON
liver,1894601,UBERON
hippocampal formation,1794600,UBERON
heart left ventricle,1764391,UBERON


# Load Uberon Ontology and Create Hyperbolic Tree Visualization

In [43]:
# Load Uberon ontology - use OWL format which is more robust
import pronto
import urllib.request

print("Loading Uberon ontology (this may take a minute)...")
try:
    # Try OWL format first (more robust)
    uberon = pronto.Ontology("http://purl.obolibrary.org/obo/uberon.owl")
    print(f"Loaded {len(uberon)} terms from Uberon ontology")
except Exception as e:
    print(f"Failed to load OWL format: {e}")
    print("Trying alternative approach...")
    
    # Alternative: download and fix the OBO file
    import tempfile
    import os
    
    obo_url = "http://purl.obolibrary.org/obo/uberon/uberon-base.obo"
    print(f"Downloading from {obo_url}")
    
    with tempfile.NamedTemporaryFile(mode='w+', suffix='.obo', delete=False) as tmp:
        tmp_path = tmp.name
    
    try:
        urllib.request.urlretrieve(obo_url, tmp_path)
        uberon = pronto.Ontology(tmp_path)
        print(f"Loaded {len(uberon)} terms from Uberon ontology")
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

Loading Uberon ontology (this may take a minute)...


/Users/rj/personal/GenePT-tools/.venv/lib/python3.10/site-packages/pronto/parsers/rdfxml.py:114: SyntaxWarning:

unknown element in `owl:ObjectProperty`: <Element '{http://purl.obolibrary.org/obo/}IAO_0000112' at 0x33a1adc60>

/Users/rj/personal/GenePT-tools/.venv/lib/python3.10/site-packages/pronto/parsers/rdfxml.py:114: SyntaxWarning:

unknown element in `owl:ObjectProperty`: <Element '{http://purl.obolibrary.org/obo/}IAO_0000112' at 0x33a1add00>

/Users/rj/personal/GenePT-tools/.venv/lib/python3.10/site-packages/pronto/parsers/rdfxml.py:114: SyntaxWarning:

unknown element in `owl:ObjectProperty`: <Element '{http://purl.obolibrary.org/obo/}IAO_0000112' at 0x33a1add50>

/Users/rj/personal/GenePT-tools/.venv/lib/python3.10/site-packages/pronto/parsers/rdfxml.py:114: SyntaxWarning:

unknown element in `owl:ObjectProperty`: <Element '{http://purl.obolibrary.org/obo/}IAO_0000116' at 0x33a1adda0>

/Users/rj/personal/GenePT-tools/.venv/lib/python3.10/site-packages/pronto/parsers/rdfxml.py:

Loaded 27208 terms from Uberon ontology


In [45]:
# Build hierarchical tree structure - more efficient approach
import numpy as np
from collections import defaultdict

print("Building tree structure from tissue data...")

# First, get tissue ontology term IDs and match with counts
tissue_ontology_ids_pdf = aggregate_obs_counts(metadata_pdf, 'tissue_ontology_term_id')
tissue_ontology_ids_pdf['tissue_name'] = tissue_counts_pdf.index
tissue_ontology_ids_pdf = tissue_ontology_ids_pdf.reset_index()
tissue_ontology_ids_pdf.columns = ['ontology_id', 'count', 'tissue_name']

# Filter for UBERON terms only
uberon_tissue_pdf = tissue_ontology_ids_pdf[tissue_ontology_ids_pdf.ontology_id.str.startswith('UBERON:')].copy()
print(f"Found {len(uberon_tissue_pdf)} tissues with UBERON ontology IDs")

# Create a mapping of ontology terms to counts
term_counts = {}
term_names = {}
for _, row in uberon_tissue_pdf.iterrows():
    term_id = row['ontology_id']
    term_counts[term_id] = row['count']
    term_names[term_id] = row['tissue_name']

print(f"Processing {len(term_counts)} tissue terms...")

# Build tree by walking up from our terms to find ancestors
tree_nodes = []
edges = []
processed = set()

def add_term_and_ancestors(term_id, max_depth=10):
    """Add a term and its ancestors to the tree"""
    if term_id in processed or term_id not in uberon or max_depth <= 0:
        return
    
    processed.add(term_id)
    term = uberon[term_id]
    
    # Add this node
    count = term_counts.get(term_id, 0)
    node = {
        'id': term_id,
        'name': term.name,
        'count': count,
        'label': term_names.get(term_id, term.name)
    }
    tree_nodes.append(node)
    
    # Get direct parents (superclasses at distance 1)
    parents = list(term.superclasses(distance=1, with_self=False))
    
    for parent in parents:
        parent_id = parent.id
        
        # Add edge from parent to this term
        if parent_id != term_id:  # Avoid self-loops
            edges.append({'parent': parent_id, 'child': term_id})
            
            # Recursively add parent
            add_term_and_ancestors(parent_id, max_depth - 1)

# Process each tissue term in our dataset
for term_id in term_counts.keys():
    add_term_and_ancestors(term_id)

print(f"Built tree with {len(tree_nodes)} nodes and {len(edges)} edges")

# Calculate levels (distance from leaves)
node_levels = {}

def calculate_level(node_id):
    """Calculate level of a node (0 = leaf)"""
    if node_id in node_levels:
        return node_levels[node_id]
    
    # Find children
    children = [e['child'] for e in edges if e['parent'] == node_id]
    
    if not children:
        level = 0  # Leaf node
    else:
        level = 1 + max(calculate_level(child) for child in children)
    
    node_levels[node_id] = level
    return level

for node in tree_nodes:
    node['level'] = calculate_level(node['id'])

tree_df = pd.DataFrame(tree_nodes)
print(f"Levels in tree: {tree_df['level'].min()} to {tree_df['level'].max()}")
print(f"\nTop 10 nodes by count:")
print(tree_df.nlargest(10, 'count')[['name', 'count', 'level']])

Building tree structure from tissue data...
Found 357 tissues with UBERON ontology IDs
Processing 357 tissue terms...
Built tree with 685 nodes and 889 edges
Levels in tree: 0 to 16

Top 10 nodes by count:
                              name     count  level
0                            blood  10841680      2
10                            lung   6166950      1
20                          breast   5633483      0
23  dorsolateral prefrontal cortex   3642741      0
27           middle temporal gyrus   3225610      0
30                 cerebral cortex   3124314      0
35                           liver   1894601      0
44           hippocampal formation   1794600      0
45            heart left ventricle   1764391      0
51                      cerebellum   1681268      0


In [46]:
# Create hyperbolic tree layout using Poincaré disk model
import math

def compute_hyperbolic_layout(tree_nodes, edges, root_id):
    """
    Compute positions for nodes in hyperbolic space (Poincaré disk).
    Uses a radial layout where depth determines distance from center.
    """
    # Build adjacency list
    children = defaultdict(list)
    for edge in edges:
        children[edge['parent']].append(edge['child'])
    
    # Position nodes
    positions = {}
    
    def layout_subtree(node_id, angle_start, angle_end, radius):
        """Recursively layout nodes in hyperbolic space"""
        positions[node_id] = (0, 0) if node_id == root_id else (
            radius * math.cos(angle_start + (angle_end - angle_start) / 2),
            radius * math.sin(angle_start + (angle_end - angle_start) / 2)
        )
        
        node_children = children.get(node_id, [])
        if not node_children:
            return
        
        # Divide angle range among children
        angle_per_child = (angle_end - angle_start) / len(node_children)
        
        # Hyperbolic spacing: increases exponentially with depth
        child_radius = min(0.95, radius + 0.2)  # Stay within unit disk
        
        for i, child_id in enumerate(node_children):
            child_angle_start = angle_start + i * angle_per_child
            child_angle_end = child_angle_start + angle_per_child
            layout_subtree(child_id, child_angle_start, child_angle_end, child_radius)
    
    # Start layout from root
    layout_subtree(root_id, 0, 2 * math.pi, 0.0)
    
    return positions

# Find root node (could be the anatomical entity or build from actual tree)
# Let's find the actual root from our edges
parent_ids = set(e['parent'] for e in edges)
child_ids = set(e['child'] for e in edges)
root_candidates = parent_ids - child_ids

if root_candidates:
    root_id = list(root_candidates)[0]
    print(f"Using root: {root_id}")
else:
    # Use a common ancestor
    root_id = "UBERON:0001062"  # anatomical entity
    print(f"Using default root: {root_id}")

print(f"Computing hyperbolic layout for {len(tree_nodes)} nodes...")
positions = compute_hyperbolic_layout(tree_nodes, edges, root_id)
print(f"Layout complete!")

Using root: BFO:0000001
Computing hyperbolic layout for 685 nodes...
Layout complete!


In [48]:
# Create the hyperbolic tree visualization
import plotly.graph_objects as go

fig = go.Figure()

# Draw edges first (so they appear behind nodes)
edge_x = []
edge_y = []
for edge in edges:
    parent_id = edge['parent']
    child_id = edge['child']
    
    if parent_id in positions and child_id in positions:
        x0, y0 = positions[parent_id]
        x1, y1 = positions[child_id]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

fig.add_trace(go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(color='lightgray', width=0.5),
    hoverinfo='none',
    showlegend=False
))

# Prepare node data
node_x = []
node_y = []
node_text = []
node_size = []
node_color = []

# Create lookup for node info
node_info = {node['id']: node for node in tree_nodes}

for node_id, (x, y) in positions.items():
    node_x.append(x)
    node_y.append(y)
    
    node = node_info.get(node_id, {})
    count = node.get('count', 0)
    name = node.get('label', node.get('name', node_id))
    
    # Create hover text
    hover_text = f"<b>{name}</b><br>"
    hover_text += f"ID: {node_id}<br>"
    hover_text += f"Count: {count:,}<br>"
    hover_text += f"Level: {node.get('level', 0)}"
    node_text.append(hover_text)
    
    # Size based on count (log scale for better visibility)
    size = 5 if count == 0 else min(30, 5 + math.log10(count + 1) * 5)
    node_size.append(size)
    
    # Color based on count (log scale)
    node_color.append(math.log10(count + 1))

# Draw nodes
fig.add_trace(go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    marker=dict(
        size=node_size,
        color=node_color,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(
            title="Log10(Count)",
            thickness=15,
            len=0.7
        ),
        line=dict(color='white', width=0.5)
    ),
    text=node_text,
    hovertemplate='%{text}<extra></extra>',
    showlegend=False
))

# Draw Poincaré disk boundary
theta = np.linspace(0, 2*np.pi, 100)
fig.add_trace(go.Scatter(
    x=np.cos(theta),
    y=np.sin(theta),
    mode='lines',
    line=dict(color='black', width=2, dash='dash'),
    hoverinfo='none',
    showlegend=False
))

# Update layout
fig.update_layout(
    title=dict(
        text="Hyperbolic Tree: Tissue Ontology (UBERON) with Cell Counts",
        x=0.5,
        xanchor='center'
    ),
    width=1000,
    height=1000,
    xaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        range=[-1.1, 1.1]
    ),
    yaxis=dict(
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        range=[-1.1, 1.1],
        scaleanchor='x',
        scaleratio=1
    ),
    plot_bgcolor='white',
    hovermode='closest'
)

fig.show()

In [59]:
# Generate standalone HTML file with d3-hypertree visualization
import json
from pathlib import Path

# Build tree from "anatomical structure" as root
def build_tree_from_root(term_counts, term_names, uberon_ontology, root_id="UBERON:0000061"):
    """Build tree from anatomical structure (UBERON:0000061) as root"""
    
    print(f"Building tree from root: {root_id}")
    if root_id in uberon_ontology:
        root_term = uberon_ontology[root_id]
        print(f"Root name: {root_term.name}")
    
    # Build parent->children map including all intermediate nodes
    children_map = defaultdict(set)
    all_relevant_terms = set()
    
    # Add all terms and their ancestors up to root
    for term_id in term_counts.keys():
        if term_id not in uberon_ontology:
            continue
        term = uberon_ontology[term_id]
        
        # Walk up to root, adding edges
        current = term_id
        visited = set()
        while current not in visited:
            visited.add(current)
            if current not in uberon_ontology:
                break
            current_term = uberon_ontology[current]
            
            parents = [p for p in current_term.superclasses(distance=1, with_self=False)]
            if not parents or current == root_id:
                break
            
            for parent in parents:
                children_map[parent.id].add(current)
                all_relevant_terms.add(parent.id)
                all_relevant_terms.add(current)
                
                if parent.id == root_id:
                    break
            
            # Continue with first parent
            if parents:
                current = parents[0].id
            else:
                break
    
    print(f"Built graph with {len(all_relevant_terms)} nodes")
    print(f"Root has {len(children_map.get(root_id, set()))} direct children")
    
    # Build hierarchical structure recursively
    def build_node(node_id, depth=0):
        if depth > 20 or node_id not in uberon_ontology:
            return None
        
        term = uberon_ontology[node_id]
        node = {
            "name": term_names.get(node_id, term.name),
            "id": node_id,
            "count": term_counts.get(node_id, 0)
        }
        
        # Add children
        if node_id in children_map and children_map[node_id]:
            children = []
            for child_id in sorted(children_map[node_id]):
                child = build_node(child_id, depth + 1)
                if child:
                    children.append(child)
            
            if children:
                node["children"] = children
        
        return node
    
    return root_id, build_node(root_id)

print("Building complete tree structure...")
root_id = "UBERON:0000061"  # anatomical structure
root_id, tree_data = build_tree_from_root(term_counts, term_names, uberon, root_id=root_id)

# Count nodes recursively
def count_nodes(node):
    if not node:
        return 0
    count = 1
    if "children" in node:
        count += sum(count_nodes(c) for c in node["children"])
    return count

print(f"Tree root: {root_id} - {tree_data['name'] if tree_data else 'None'}")
print(f"Total nodes in tree: {count_nodes(tree_data)}")

# Create HTML file with dramatically scaled node points
html_content = '''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>UBERON Tissue Ontology - Hyperbolic Tree</title>
    <link href="https://cdn.jsdelivr.net/npm/d3-hypertree@1.1.3/dist/d3-hypertree-light.min.css" rel="stylesheet">
    <style>
        body {
            margin: 0;
            padding: 20px;
            font-family: Arial, sans-serif;
            background: #f5f5f5;
        }
        h1 {
            text-align: center;
            color: #333;
            margin-bottom: 10px;
        }
        .info {
            text-align: center;
            color: #666;
            margin-bottom: 20px;
            font-size: 14px;
        }
        #hypertree {
            width: 100%;
            height: 900px;
            background: white;
            border: 1px solid #ddd;
            border-radius: 4px;
        }
        .legend {
            text-align: center;
            color: #666;
            margin-top: 10px;
            font-size: 12px;
        }
    </style>
</head>
<body>
    <h1>UBERON Tissue Ontology - Hyperbolic Tree Visualization</h1>
    <div class="info">Click nodes to focus | Drag to pan | Scroll to zoom | Node size = cell count</div>
    <div id="hypertree"></div>
    <div class="legend">Node size and color represent the number of cells in the dataset (log scale)</div>
    
    <script src="https://d3js.org/d3.v7.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/d3-hypertree@1.1.3/dist/d3-hypertree.min.js"></script>
    <script>
        const data = ''' + json.dumps(tree_data, indent=2) + ''';
        
        console.log('Tree root:', data.name);
        console.log('Children count:', data.children ? data.children.length : 0);
        
        // Initialize hypertree
        const ht = new hyt.Hypertree(
            { parent: document.getElementById('hypertree') },
            {
                dataloader: ok => ok(data),
                langInitBFS: (ht, n) => {
                    n.precalc.label = n.data.name || n.data.id;
                    n.precalc.clickable = true;
                },
                objects: {
                    node: {
                        fill: (n) => {
                            const count = n.data.count || 0;
                            if (count === 0) return '#cccccc';
                            // Use viridis colors based on count
                            const intensity = Math.min(1, Math.log10(count) / 7);
                            const t = intensity;
                            const r = Math.floor(255 * (0.267 + 0.004 * t + 0.329 * t * t));
                            const g = Math.floor(255 * (0.004 + 0.784 * t - 0.224 * t * t));
                            const b = Math.floor(255 * (0.329 + 0.520 * t - 0.133 * t * t));
                            return `rgb(${r},${g},${b})`;
                        },
                        r: (n) => {
                            const count = n.data.count || 0;
                            if (count === 0) return 0.002;  // Very small for nodes without data
                            
                            // Dramatically scale node size based on log of count
                            // Range from 0.005 (10 cells) to 0.06 (10M+ cells)
                            const logCount = Math.log10(count + 1);
                            
                            // More aggressive scaling: 0.005 + (logCount * 0.008)
                            // log10(100) = 2 -> 0.021
                            // log10(10000) = 4 -> 0.037
                            // log10(1000000) = 6 -> 0.053
                            // log10(10000000) = 7 -> 0.061
                            return Math.min(0.06, 0.005 + logCount * 0.008);
                        },
                        stroke: 'white',
                        'stroke-width': 0.0008,
                    },
                    link: {
                        stroke: '#dddddd',
                        'stroke-width': 0.0005,
                    },
                    label: {
                        fill: '#333333',
                        'font-size': '8px',
                    },
                },
                interaction: {
                    λbounds: [0.1, 0.9],
                }
            }
        );
        
        ht.initPromise
            .then(() => new Promise((ok, err) => ht.animateUp(ok, err)))
            .then(() => ht.drawDetailFrame());
    </script>
</body>
</html>'''

# Save HTML file
output_path = Path('hyperbolic_tree_uberon.html').absolute()
output_path.write_text(html_content)

print(f"\n✓ HTML file saved successfully!")
print(f"\nFile location: {output_path}")
print(f"\nTo view: open {output_path}")

Building complete tree structure...
Building tree from root: UBERON:0000061
Root name: anatomical structure
Built graph with 665 nodes
Root has 11 direct children
Tree root: UBERON:0000061 - anatomical structure
Total nodes in tree: 762

✓ HTML file saved successfully!

File location: /Users/rj/personal/GenePT-tools/notebooks/hyperbolic_tree_uberon.html

To view: open /Users/rj/personal/GenePT-tools/notebooks/hyperbolic_tree_uberon.html
